# Load dim_region

dim_region is derived from bronze.products data.

Starting the notebook with %run "/Workspace/Shared/notebook_init"  loads common variables and constants used across all notebooks for the Vinoworld project

CATALOG, BRONZE, SILVER, GOLD, AUDIT, RAW_FILES,
PIPELINE_RUN_ID, Utils, F, Row, datetime etc.

import uuid, time
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit, input_file_name, col
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, IntegerType
sys.path.append("/Workspace/Shared")

import pipeline_utils as Utils
from pipeline_logging import pipeline_log_upsert, pipeline_step_log_upsert, ingestion_log_insert


In [0]:
%run "../../libs/notebook_init"

In [0]:
# Imports and constants specific to the Arancione Bronze load.
# STORE_NAME identifies the source store; SOURCE_SUBPATH is the subfolder
# under RAW_FILES; TARGET_TABLE is the fully-qualified Bronze table name.

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime, timezone
import traceback

%load_ext autoreload
%autoreload 2

SOURCE_SUBPATH = "masterdata"                       # case must match volume
SOURCE_PATH    = f"{RAW_FILES}{SOURCE_SUBPATH}"
TARGET_TABLE   = f"{SILVER}.dim_region"

# print(f"Products Source Path:  {SOURCE_PATH}")


In [0]:
# ----------------------------------------------------------------------
# Setup the variables need for initial call to pipeline_step_log_upsert
# ----------------------------------------------------------------------

nb = Utils.get_notebook_context(dbutils)
notebook_folder = nb['notebook_folder']
notebook_name = nb['notebook_name']

#logger.info(f"Inserting pipeline_step_log record for notebook {notebook_name}")

step_log_id       = str(uuid.uuid4())
pipeline_run_id   = PIPELINE_RUN_ID
step_sequence     = 1
layer             = "silver"
target_table      = TARGET_TABLE
status            = "running"
started_timestamp = datetime.now(timezone.utc)
rows_read         = 0
rows_written      = 0
error_message     = None


pipeline_step_log_upsert(spark, step_log_id, pipeline_run_id, step_sequence, notebook_folder, notebook_name, status, started_timestamp, layer, target_table)

# All Parameters
# pipeline_step_log_upsert(spark, step_log_id, pipeline_run_id, step_sequence, notebook_folder, notebook_name, status, started_timestamp, layer, target_table, rows_read, rows_written,  ended_timestamp,  error_message)

In [0]:
%skip
%sql

/* -----------------------------------------------------------------------
-- The raw datafiles for these are in pristine condition and can be reloaded without any processing
----------------------------------------------------------------------- */

TRUNCATE TABLE vinoworld.silver.dim_region;


In [0]:
%skip


metrics = spark.sql(f"DESCRIBE HISTORY {CATALOG}.silver.dim_product LIMIT 1") \
               .select("operationMetrics") \
               .collect()[0][0]

rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
rows_updated  = int(metrics.get("numTargetRowsUpdated",  0))
rows_deleted  = int(metrics.get("numTargetRowsDeleted",  0))

print(f" Rows  Inserted: {rows_inserted:,}")
print(f" Rows     Updated: {rows_updated:,}"   )


In [0]:
%skip
%sql
select count(*) from vinoworld.silver.dim_region